In [0]:
%sql
MERGE INTO la_lakehouse.gold.dim_zone AS target
USING (
    SELECT zone, zone_base 
    FROM la_lakehouse.silver.silver_permits_issued_enriched 
    WHERE zone_base IS NOT NULL
    GROUP BY zone, zone_base
) AS source
ON target.zone_base = source.zone_base
WHEN NOT MATCHED THEN
  INSERT (zone, zone_base) 
  VALUES (source.zone, source.zone_base);


In [0]:
%sql
MERGE INTO la_lakehouse.gold.dim_date AS target
USING (
    WITH smallest_date AS (
        SELECT LEAST(MIN(submitted_date), MIN(issue_date), MIN(cofo_date)) AS start_date 
        FROM la_lakehouse.silver.silver_permits_issued_enriched
    ), 
    largest_date AS (
        SELECT GREATEST(MAX(submitted_date), MAX(issue_date), MAX(cofo_date)) AS end_date 
        FROM la_lakehouse.silver.silver_permits_issued_enriched
    ),
    date_sequence AS (
        SELECT explode(sequence(s.start_date, l.end_date + interval 1 year, interval 1 day)) AS full_date
        FROM smallest_date AS s, largest_date AS l
    )
    SELECT 
        d.full_date,
        YEAR(d.full_date) AS year,
        MONTH(d.full_date) AS month,
        DAY(d.full_date) AS day,
        DATE_FORMAT(d.full_date, 'E') AS day_name,
        DAYOFWEEK(d.full_date) AS day_of_week,
        DATE_FORMAT(d.full_date, 'MMMM') AS month_name
    FROM date_sequence d
) AS source
ON target.full_date = source.full_date
WHEN NOT MATCHED THEN
  INSERT (full_date, year, month, day, day_name, day_of_week, month_name)
  VALUES (source.full_date, source.year, source.month, source.day, source.day_name, source.day_of_week, source.month_name);


In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_census_tract AS target
USING (
    SELECT ct
    FROM la_lakehouse.silver.silver_permits_issued_enriched 
    WHERE ct IS NOT NULL
    GROUP BY ct
) AS source
ON target.ct = source.ct
WHEN NOT MATCHED THEN 
    INSERT(ct)
    VALUES(source.ct);

In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_district AS target
USING (
    SELECT cd
    FROM la_lakehouse.silver.silver_permits_issued_enriched 
    WHERE cd IS NOT NULL
    GROUP BY cd
) AS source
ON target.cd = source.cd
WHEN NOT MATCHED THEN 
    INSERT(cd)
    VALUES(source.cd);

In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_area_planning_commision AS target
USING (
    SELECT apc
    FROM la_lakehouse.silver.silver_permits_issued_enriched 
    WHERE apc IS NOT NULL
    GROUP BY apc
) AS source
ON target.apc = source.apc
WHEN NOT MATCHED THEN 
    INSERT(apc)
    VALUES(source.apc);

In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_community_plan_area AS target
USING (
    SELECT cpa
    FROM la_lakehouse.silver.silver_permits_issued_enriched 
    WHERE cpa IS NOT NULL
    GROUP BY cpa
) AS source
ON target.cpa = source.cpa
WHEN NOT MATCHED THEN 
    INSERT(cpa)
    VALUES(source.cpa);

In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_certified_neighborhood_council AS target
USING (
    SELECT cnc
    FROM la_lakehouse.silver.silver_permits_issued_enriched
    WHERE cnc IS NOT NULL
    GROUP BY cnc
) AS source
ON target.cnc = source.cnc
WHEN NOT MATCHED THEN 
    INSERT(cnc)
    VALUES(source.cnc);

In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_hillside_ordinance_area AS target
USING (
    SELECT hl
    FROM la_lakehouse.silver.silver_permits_issued_enriched
    WHERE hl IS NOT NULL
    GROUP BY hl
) AS source
ON target.hl = source.hl
WHEN NOT MATCHED THEN 
    INSERT(hl)
    VALUES(source.hl);

In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_permit_type AS target
USING (
    SELECT 
    permit_group,
    permit_type,
    permit_sub_type
    FROM la_lakehouse.silver.silver_permits_issued_enriched
    WHERE permit_group IS NOT NULL 
    AND permit_type IS NOT NULL
    GROUP BY permit_group,permit_type,permit_sub_type
) AS source
ON target.permit_group = source.permit_group
AND target.permit_type = source.permit_type
AND target.permit_sub_type = source.permit_sub_type
WHEN NOT MATCHED THEN 
    INSERT(permit_group,permit_type,permit_sub_type)
    VALUES(
        source.permit_group,
        source.permit_type,
        source.permit_sub_type
        );


In [0]:
%sql 
MERGE INTO la_lakehouse.gold.dim_use_type AS target
USING (
    SELECT 
    use_code,
    use_desc
    FROM la_lakehouse.silver.silver_permits_issued_enriched
    WHERE use_code IS NOT NULL
    AND use_desc IS NOT NULL
    GROUP BY use_code,use_desc
) AS source
ON target.use_code = source.use_code
AND target.use_desc = source.use_desc
WHEN NOT MATCHED THEN 
    INSERT(use_code,use_desc)
    VALUES(
        source.use_code,
        source.use_desc
        );


#Testing Populated Tables 

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_zone;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_permit_type;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_hillside_ordinance_area;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_certified_neighborhood_council;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_community_plan_area;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_district;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_area_planning_commision;

In [0]:
%sql
SELECT COUNT(*)
FROM la_lakehouse.gold.dim_census_tract;

In [0]:
%sql
SELECT MIN(full_date), MAX(full_date)
FROM la_lakehouse.gold.dim_date;


In [0]:
%sql
SELECT full_date
FROM la_lakehouse.gold.dim_date
WHERE full_date = '2026-08-16'

#Testing For Duplicates

In [0]:
%sql

SELECT zone_base, COUNT(*)
FROM la_lakehouse.gold.dim_zone
GROUP BY zone_base
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT ct, COUNT(*)
FROM la_lakehouse.gold.dim_census_tract
GROUP BY ct
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT cd, COUNT(*)
FROM la_lakehouse.gold.dim_district
GROUP BY cd
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT apc, COUNT(*)
FROM la_lakehouse.gold.dim_area_planning_commision
GROUP BY apc
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT cpa, COUNT(*)
FROM la_lakehouse.gold.dim_community_plan_area
GROUP BY cpa
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT cnc, COUNT(*)
FROM la_lakehouse.gold.dim_certified_neighborhood_council
GROUP BY cnc
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT hl, COUNT(*)
FROM la_lakehouse.gold.dim_hillside_ordinance_area
GROUP BY hl
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT 
permit_group,
permit_type,
permit_sub_type, 
COUNT(*)
FROM la_lakehouse.gold.dim_permit_type
GROUP BY permit_group, permit_type, permit_sub_type
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT 
use_code,
use_desc,
COUNT(*)
FROM la_lakehouse.gold.dim_use_type
GROUP BY use_code, use_desc
HAVING COUNT(*) > 1;

In [0]:
%sql

SELECT full_date, COUNT(*)
FROM la_lakehouse.gold.dim_date
GROUP BY full_date
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT permit_nbr, COUNT(*)
FROM la_lakehouse.silver.silver_permits_issued_enriched
GROUP BY permit_nbr
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa ON s.hl = dhoa.hl
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa ON s.hl = dhoa.hl
LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt 
    ON s.permit_group = dpt.permit_group 
    AND s.permit_type = dpt.permit_type
    AND s.permit_sub_type = dpt.permit_sub_type
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa ON s.hl = dhoa.hl
LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt 
    ON s.permit_group = dpt.permit_group 
    AND s.permit_type = dpt.permit_type
    AND s.permit_sub_type = dpt.permit_sub_type
LEFT JOIN la_lakehouse.gold.dim_use_type AS dut 
    ON s.use_code = dut.use_code
    AND s.use_desc = dut.use_desc
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT use_code, use_desc, COUNT(*) 
FROM la_lakehouse.gold.dim_use_type
GROUP BY use_code, use_desc
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT use_code, use_desc, COUNT(*) 
FROM la_lakehouse.gold.dim_use_type
WHERE use_code = 23
GROUP BY use_code, use_desc
ORDER BY use_code;

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa ON s.hl = dhoa.hl
LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt ON s.permit_type = dpt.permit_type AND s.permit_sub_type = dpt.permit_sub_type AND s.permit_group = dpt.permit_group
LEFT JOIN la_lakehouse.gold.dim_use_type AS dut ON s.use_code = dut.use_code AND s.use_desc = dut.use_desc
LEFT JOIN la_lakehouse.gold.dim_date AS issue_date ON s.issue_date = issue_date.full_date
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa ON s.hl = dhoa.hl
LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt ON s.permit_type = dpt.permit_type AND s.permit_sub_type = dpt.permit_sub_type AND s.permit_group = dpt.permit_group
LEFT JOIN la_lakehouse.gold.dim_use_type AS dut ON s.use_code = dut.use_code AND s.use_desc = dut.use_desc
LEFT JOIN la_lakehouse.gold.dim_date AS issue_date ON s.issue_date = issue_date.full_date
LEFT JOIN la_lakehouse.gold.dim_date AS submitted_date ON s.submitted_date = submitted_date.full_date
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql
SELECT COUNT(*) 
FROM la_lakehouse.silver.silver_permits_issued_enriched AS s
LEFT JOIN la_lakehouse.gold.dim_zone AS dz ON s.zone_base = dz.zone_base
LEFT JOIN la_lakehouse.gold.dim_census_tract AS dct ON s.ct = dct.ct
LEFT JOIN la_lakehouse.gold.dim_district AS dd ON s.cd = dd.cd
LEFT JOIN la_lakehouse.gold.dim_area_planning_commision AS dapc ON s.apc = dapc.apc
LEFT JOIN la_lakehouse.gold.dim_community_plan_area AS dcpa ON s.cpa = dcpa.cpa
LEFT JOIN la_lakehouse.gold.dim_certified_neighborhood_council AS dnc ON s.cnc = dnc.cnc
LEFT JOIN la_lakehouse.gold.dim_hillside_ordinance_area AS dhoa ON s.hl = dhoa.hl
LEFT JOIN la_lakehouse.gold.dim_permit_type AS dpt ON s.permit_type = dpt.permit_type AND s.permit_sub_type = dpt.permit_sub_type AND s.permit_group = dpt.permit_group
LEFT JOIN la_lakehouse.gold.dim_use_type AS dut ON s.use_code = dut.use_code AND s.use_desc = dut.use_desc
LEFT JOIN la_lakehouse.gold.dim_date AS issue_date ON s.issue_date = issue_date.full_date
LEFT JOIN la_lakehouse.gold.dim_date AS submitted_date ON s.submitted_date = submitted_date.full_date
LEFT JOIN la_lakehouse.gold.dim_date AS cofo_date ON s.cofo_date =   
cofo_date.full_date
WHERE s.permit_nbr = '26048-10000-00213';

In [0]:
%sql

SELECT COUNT(*)
FROM la_lakehouse.gold.fact_permits